In [ ]:
import tensorflow as tf
from keras import layers, Model
import numpy as np

(train, _), _ = tf.keras.datasets.mnist.load_data()
train = train.astype('float32')/255.0
train = np.expand_dims(train, 1)

latent_dim = 2

# generator 
generator = tf.keras.Sequential([
    layers.Input(7*7*128, input_shape=(latent_dim, 1)),
    layers.Flatten(),
    layers.Conv2DTranspose(128, 'same', activation='relu'),
    layers.Dense(1, 4, 2, activation='sigmoid')
])

# discriminator
discriminator = tf.keras.Sequential([
    layers.Input(28*28),
    layers.Conv2D(64, 'same', activation='relu'),
    layers.Dense(28*28, activation='sigmoid')
])
discriminator.compile(optimizer='adam', loss='binary_crossentropy')
discriminator.trainable = False 

# gan
input_gan = layers.Input(input_shape=(latent_dim, 1))
gan = Model(discriminator(generator(input_gan)))

epochs = 3 
batch_size = 64 
steps =  (batch_size//epochs)

for step in steps:
    noise = np.random.normal((batch_size, 1))
    gen_img = generator(noise)
    idx = np.random.randint(0, train[0], batch_size)
    real_imgs = train[idx]

    images = [real_imgs, gen_img]
    labels = [np.ones((batch_size, ), np.zeors(batch_size, ))]
    discriminator_loss = discriminator.train_on_batch(images, labels)

    noise = np.random.normal((batch_size, 1))
    gan_loss = gan.train_on_batch(generator(noise), np.ones())